# Lab 25-01 · Oxidation, Gate Trenches, Wire Tiers and Fab Flow

**Predict first**, then run. The handout is [README.md](README.md).

In [1]:
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import numpy as np
import process as p

## Part 1 · Oxidation

*Predict* whether 1 h of wet oxidation at 1000 °C on bare (100) silicon grows more or less than 250 nm.

In [2]:
x_cf = p.oxide_thickness(1.0, 1000, "wet")
x_ode = p.oxide_ode(1.0, 1000, "wet")
print(f"closed form {x_cf*1000:.1f} nm, ODE {x_ode*1000:.1f} nm, difference {abs(x_cf-x_ode)*1e3*1e3:.4f} pm")
print(f"dry, 1 h at 1000 C from xi = 25 nm: {p.oxide_thickness(1.0, 1000, 'dry', 0.025)*1000:.0f} nm")
t = np.logspace(-2, 2, 200)
plt.figure()
for amb, ls in (("wet", "-"), ("dry", "--")):
    for tc in (900, 1000, 1100):
        plt.loglog(t, [p.oxide_thickness(ti, tc, amb) for ti in t], ls, label=f"{amb} {tc} C")
plt.xlabel("time (h)"); plt.ylabel("oxide thickness (um)"); plt.legend(fontsize=7); plt.grid(True, which="both")
plt.show()

closed form 388.4 nm, ODE 388.4 nm, difference 0.0000 pm
dry, 1 h at 1000 C from xi = 25 nm: 58 nm


## Part 2 · Gate trench

*Predict* the narrowest trench that leaves 3.0 nm of fill with 1.8 nm HfO₂ and the 4.5 nm work-function flavour.

In [3]:
widths = np.linspace(10, 22, 121)
for twf in (3.0, 4.5):
    plt.plot(widths, [max(p.fill_width(w, 1.8, twf), 0) for w in widths], label=f"work-function {twf} nm")
w3 = 3.0 + 2 * (1.8 + 4.5)
print(f"narrowest trench for 3 nm fill with the 4.5 nm flavour: {w3:.1f} nm")
plt.axhline(3.0, color="0.5", ls=":")
plt.xlabel("trench width (nm)"); plt.ylabel("fill width (nm)"); plt.legend(); plt.grid(True)
plt.show()

narrowest trench for 3 nm fill with the 4.5 nm flavour: 15.6 nm


## Part 3 · Wire tiers

*Predict* the length at which the global tier's RC delay equals its time of flight.

In [4]:
tiers = {"local": (20, 40, 2, 4.5), "intermediate": (40, 80, 2, 2.8), "global": (400, 800, 2, 1.9)}
L = np.logspace(1, 4, 200)  # um
plt.figure()
for name, (w, h, tb, rho) in tiers.items():
    r = p.wire_r_per_um(w, h, tb, rho)
    plt.loglog(L / 1000, [p.rc_delay(r, 0.2e-15, l) * 1e12 for l in L], label=name)
    print(f"{name:12s} {r:8.4f} ohm/um, 1 mm delay {p.rc_delay(r, 0.2e-15, 1000)*1e12:9.2f} ps")
plt.loglog(L / 1000, [p.time_of_flight(l, 3.0) * 1e12 for l in L], "k:", label="time of flight")
rg = p.wire_r_per_um(400, 800, 2, 1.9)
print(f"global RC equals time of flight at {p.rc_equals_tof_length_um(rg, 0.2e-15, 3.0)/1000:.2f} mm")
plt.xlabel("length (mm)"); plt.ylabel("delay (ps)"); plt.legend(); plt.grid(True, which="both")
plt.show()

local         74.0132 ohm/um, 1 mm delay   5606.46 ps
intermediate   9.9715 ohm/um, 1 mm delay    755.34 ps
global         0.0601 ohm/um, 1 mm delay      4.55 ps
global RC equals time of flight at 1.27 mm


## Part 4 · Little's law

*Predict* the average WIP of a fab starting 100,000 wafers a month with a 12-week mean cycle time.

In [5]:
lam = 100000 / p.DAYS_PER_MONTH  # wafers per day
T = 84.0                          # days
sim = p.simulate_wip(lam, T)
print(f"lambda*T = {lam*T:,.0f} wafers; simulated average WIP = {sim:,.0f} ({sim/(lam*T):.4f} of lambda*T)")

lambda*T = 275,975 wafers; simulated average WIP = 276,012 (1.0001 of lambda*T)
